# rackfocus — ingest offline trên Kaggle

Video → **shot** (TransNetV2) → **7 keyframe/shot** (webp) → **scene** (BaSSL CRN)
→ **transcript** (chunkformer) → **cắt clip scene** → **upload S3**.

Khác pipeline trong `services/ingest`:

- Ghi **7 keyframe/shot** thay vì 3. BaSSL vẫn chỉ ăn **3 frame** (start/mid/end) nên
  kết quả gộp scene không xê dịch — xem cell *Keyframe*.
- Output tách thành 3 cây song song theo group thay vì 1 thư mục phẳng mỗi video.
- Không embed SigLIP / không ghi parquet (cần Postgres cấp `video_id`, Kaggle không có).

## Cần chuẩn bị trước

| Thứ | Lấy ở đâu | Vì sao |
|---|---|---|
| `inference/` của TransNetV2 | repo [soCzech/TransNetV2](https://github.com/soCzech/TransNetV2) | chứa `transnetv2.py`, import bằng PYTHONPATH |
| `transnetv2-weights/` | cùng repo trên | weights đã train, không có thì mạng ra rác |
| `bassl.ckpt` | checkpoint BaSSL của bạn | **bắt buộc** — random weights cho scene là vô nghĩa |
| Internet | Settings → Internet | để `pip install` chunkformer từ git |
| GPU | Settings → Accelerator | TransNetV2 + ResNet50 + ASR |

Upload 3 thứ đầu thành **Kaggle Dataset** rồi attach, vì Kaggle mặc định tắt internet.
AWS key để trong **Add-ons → Secrets**, đừng hardcode.

## Cấu trúc output

```
Keyframes_L21_a/keyframes/L21_V001/{000012.webp, shots.csv, keyframes.json}
Keyscence_L21_a/keyscence/L21_V001/{scene_000.mp4, scenes.json}
Transcripts_L21_a/transcripts/L21_V001.json
```

## 1. Config

Chỉ cell này cần sửa. Mọi cell dưới đọc biến từ đây.

In [ ]:
import glob
import os

# ── S3 ───────────────────────────────────────────────────────────────
AWS_REGION      = "ap-southeast-1"
AWS_BUCKET_NAME = "aic-bucket-2026"

# Ưu tiên Kaggle Secrets (Add-ons -> Secrets). Điền tay vào đây nếu chạy ngoài Kaggle.
AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient

    _sec = UserSecretsClient()
    AWS_ACCESS_KEY = AWS_ACCESS_KEY or _sec.get_secret("AWS_ACCESS_KEY")
    AWS_SECRET_KEY = AWS_SECRET_KEY or _sec.get_secret("AWS_SECRET_KEY")
except Exception as ex:
    print(f"! Chưa lấy được Kaggle Secrets ({type(ex).__name__}) — điền tay nếu cần upload")

# Base URL ghi vào keyframe_url / scene_url. Gắn CloudFront sau thì chỉ sửa dòng này.
S3_URL_BASE = f"https://{AWS_BUCKET_NAME}.s3.{AWS_REGION}.amazonaws.com"

DO_UPLOAD                 = False   # True = đẩy lên S3 sau mỗi video
DELETE_LOCAL_AFTER_UPLOAD = False   # True = xoá local sau upload (đĩa Kaggle ~20GB)

# ── Checkpoint (ASR không cần: nạp theo tên HF) ──────────────────────
TRANSNET_CODE    = "/kaggle/input/transnetv2/inference"          # chứa transnetv2.py
TRANSNET_WEIGHTS = "/kaggle/input/transnetv2/transnetv2-weights"
BASSL_CKPT       = "/kaggle/input/bassl/bassl.ckpt"
ASR_MODEL        = "khanhld/chunkformer-ctc-large-vie"

# ── GPU ──────────────────────────────────────────────────────────────
# TransNetV2 (thư mục inference/) chạy TensorFlow; ASR + ResNet50 + BaSSL chạy
# PyTorch. TF mặc định chiếm gần hết VRAM của MỌI GPU nó thấy và KHÔNG tự bật
# memory growth (đã đọc inference/transnetv2.py: không có set_memory_growth /
# set_visible_devices) -> PyTorch nạp sau sẽ CUDA OOM. Biến này phải đặt TRƯỚC
# khi TF khởi tạo, nên nó nằm ở cell config chứ không phải cell nạp model.
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

TORCH_GPU = 0      # GPU cho PyTorch (ASR, ResNet50, BaSSL)
TF_GPU    = None   # None = TF dùng chung GPU với PyTorch (đã bật allow_growth).
                   # Có 2 GPU (Kaggle T4 x2): đặt 1 để TF chiếm riêng GPU 1,
                   # tách hoàn toàn khỏi PyTorch -> hết rủi ro tranh VRAM.

# Chia việc cho nhiều session chạy song song: session A (0, 2), session B (1, 2).
# Đây là cách thực tế để dùng hết 2 GPU — xem mục "2 GPU" trong cell markdown dưới.
SHARD, NUM_SHARDS = 0, 1

# ── Chạy ─────────────────────────────────────────────────────────────
INPUT_ROOT      = "/kaggle/input"
OUT_ROOT        = "/kaggle/working/out"
N_KEYFRAMES     = 7      # số keyframe ghi ra đĩa mỗi shot
INSET           = 0.12   # lùi vào 12% mỗi đầu shot, tránh frame fade/dissolve
KEYFRAME_HEIGHT = 360    # 360 để tái dùng cho encoder ảnh (input 384)
WEBP_QUALITY    = 90
SHOT_THRESHOLD  = 0.5    # hạ 0.3-0.4 nếu sót gradual cut
SCENE_THRESHOLD = 0.55
LIMIT           = 1      # 0 = tất cả. Để 1 khi chạy thử.
OVERWRITE       = False

# 3 frame BaSSL dùng = start / mid / end trong dãy N_KEYFRAMES.
# N phải LẺ: chỉ khi đó mới có 1 frame nằm đúng giữa shot, trùng mid của công thức
# 3-frame gốc. N chẵn -> mid lệch ở ~98% shot -> BaSSL gộp scene khác bản gốc.
assert N_KEYFRAMES % 2 == 1, f"N_KEYFRAMES phải lẻ (đang {N_KEYFRAMES}) — xem cell Keyframe"
BASSL_KF_COLS = tuple(f"kf{i}" for i in (0, (N_KEYFRAMES - 1) // 2, N_KEYFRAMES - 1))

os.environ.update(
    AWS_ACCESS_KEY=AWS_ACCESS_KEY or "",
    AWS_SECRET_KEY=AWS_SECRET_KEY or "",
    AWS_REGION=AWS_REGION,
    AWS_BUCKET_NAME=AWS_BUCKET_NAME,
)

print(f"out       = {OUT_ROOT}")
print(f"keyframe  = {N_KEYFRAMES}/shot, BaSSL đọc {BASSL_KF_COLS}")
print(f"upload    = {DO_UPLOAD} -> {S3_URL_BASE}")
print(f"limit     = {LIMIT or 'tất cả'}")

## 2. Cài đặt

Image Kaggle có sẵn phần lớn (torch, tensorflow, pandas, PIL, boto3, ffmpeg) nên cell
này **chỉ cài thứ còn thiếu**. `chunkformer` không có trên PyPI, phải cài từ git →
**cần bật Internet** trong Settings.

Chạy trước Preflight, để Preflight phản ánh trạng thái sau khi cài.

In [ ]:
import importlib
import shutil
import subprocess
import sys


def _has(mod):
    try:
        importlib.import_module(mod)
        return True
    except ImportError:
        return False


def _pip(*args):
    print(f"  cài: {' '.join(args)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)


# (tên module để import, tên package để pip). Chỉ cài cái thiếu -> rerun nhanh.
for mod, pkg in (
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("cv2", "opencv-python-headless"),
    ("PIL", "pillow"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("tensorflow", "tensorflow"),   # TransNetV2 thư mục inference/ chạy TF, không phải torch
    ("boto3", "boto3"),
):
    if _has(mod):
        print(f"  có sẵn: {mod}")
    else:
        _pip(pkg)

# chunkformer không có trên PyPI.
if _has("chunkformer"):
    print("  có sẵn: chunkformer")
else:
    _pip("git+https://github.com/khanld/chunkformer.git")

# ffmpeg là binary hệ thống, không phải pip.
if shutil.which("ffmpeg"):
    print("  có sẵn: ffmpeg")
else:
    print("  cài: ffmpeg (apt)")
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=False)

# TransNetV2 import theo PYTHONPATH, không phải package pip.
if TRANSNET_CODE not in sys.path:
    sys.path.insert(0, TRANSNET_CODE)
print("\nsys.path[0] =", sys.path[0])

## 3. Preflight

Kiểm tra trước khi tốn giờ GPU. Thiếu checkpoint là fail luôn ở đây thay vì chết
giữa batch.

In [ ]:
import shutil


def _check(ok, label, hint=""):
    print(("  OK   " if ok else "  MISS ") + label + ("" if ok else f"\n         -> {hint}"))
    return bool(ok)


problems = []

for exe in ("ffmpeg", "ffprobe"):
    if not _check(shutil.which(exe), exe, "apt-get install -y ffmpeg (cần Internet)"):
        problems.append(exe)

try:
    import torch

    _check(True, f"torch {torch.__version__}")
    has_cuda = torch.cuda.is_available()
    n = torch.cuda.device_count() if has_cuda else 0
    _check(has_cuda,
           f"GPU x{n}: {', '.join(torch.cuda.get_device_name(i) for i in range(n)) if has_cuda else 'không có, sẽ chạy CPU (rất chậm)'}",
           "Settings -> Accelerator -> GPU")
    if n > 1:
        print(f"         {n} GPU: đặt TF_GPU=1 để tách TensorFlow khỏi PyTorch, "
              f"và/hoặc dùng SHARD/NUM_SHARDS cho nhiều session")
except ImportError:
    problems.append("torch")
    _check(False, "torch", "pip install torch")

try:
    import tensorflow as tf

    _check(True, f"tensorflow {tf.__version__} (TransNetV2 cần TF, không phải PyTorch)")
    _check(os.environ.get("TF_FORCE_GPU_ALLOW_GROWTH") == "true",
           "TF_FORCE_GPU_ALLOW_GROWTH=true",
           "chạy lại cell Config TRƯỚC khi import TF, nếu không TF chiếm hết VRAM")
except ImportError:
    problems.append("tensorflow")
    _check(False, "tensorflow",
           "pip install tensorflow — TransNetV2 inference/ là TF. Image Kaggle "
           "thường có sẵn; requirements.txt của repo thì KHÔNG khai TF.")

for label, path, hint in (
    ("TRANSNET_CODE", TRANSNET_CODE, "thư mục inference/ của repo soCzech/TransNetV2"),
    ("TRANSNET_WEIGHTS", TRANSNET_WEIGHTS, "thư mục transnetv2-weights/ cùng repo"),
    ("BASSL_CKPT", BASSL_CKPT, "file .ckpt của BaSSL — bắt buộc, không chạy random weights"),
):
    if not _check(os.path.exists(path), f"{label} = {path}", hint):
        problems.append(label)

free_gb = shutil.disk_usage(os.path.dirname(OUT_ROOT) or "/kaggle/working").free / 2**30
_check(free_gb > 5, f"đĩa trống {free_gb:.1f} GB",
       "bật DELETE_LOCAL_AFTER_UPLOAD nếu group lớn")

videos_found = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "video", "*.mp4"), recursive=True))
if not videos_found:
    videos_found = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "*.mp4"), recursive=True))
_check(videos_found, f"tìm thấy {len(videos_found)} video",
       f"kiểm tra INPUT_ROOT={INPUT_ROOT} và cấu trúc Videos_*/video/*.mp4")
for v in videos_found[:3]:
    print(f"         {v}")

print()
print("THIẾU: " + ", ".join(problems) if problems else "Preflight OK — chạy tiếp được")

## 4. Stage: probe + shot detect

TransNetV2 giữ toàn bộ frame 48×27 trong RAM khi `predict_video`, nên phải giải
phóng ngay sau mỗi video, không thì RAM leo dần qua cả batch.

In [ ]:
import subprocess


def probe(path):
    """Trả (width, height, fps, n_frames). n_frames = -1 khi container không ghi nb_frames."""
    out = subprocess.run(
        ["ffprobe", "-v", "quiet", "-select_streams", "v:0",
         "-show_entries", "stream=width,height,r_frame_rate,nb_frames",
         "-of", "default=nw=1:nk=1", path],
        capture_output=True, text=True, check=True,
    ).stdout.split()

    w, h = int(out[0]), int(out[1])
    num, den = out[2].split("/")
    fps = float(num) / float(den)
    try:
        n = int(out[3])
    except (IndexError, ValueError):
        n = -1
    return w, h, fps, n


def configure_tf_gpu(gpu_index=None):
    """Bật memory growth cho TF, tuỳ chọn ghim TF vào 1 GPU.

    PHẢI gọi trước khi TF cấp phát VRAM lần đầu. Không làm việc này thì TF chiếm
    gần hết VRAM của mọi GPU visible và PyTorch nạp sau sẽ CUDA OOM.
    """
    import tensorflow as tf

    gpus = tf.config.list_physical_devices("GPU")
    if not gpus:
        print("    TF: không thấy GPU -> TransNetV2 chạy CPU (chậm)")
        return
    try:
        if gpu_index is not None and gpu_index < len(gpus):
            tf.config.set_visible_devices([gpus[gpu_index]], "GPU")
            gpus = [gpus[gpu_index]]
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
        print(f"    TF: dùng {[g.name.split('/')[-1] for g in gpus]}, allow_growth=True")
    except RuntimeError as ex:
        # TF đã cấp phát rồi -> không đổi được nữa trong process này.
        print(f"    ! TF đã init, không đổi được GPU config: {ex}\n"
              f"      Nếu gặp CUDA OOM: Run -> Restart & clear cell outputs, chạy lại từ đầu.")


def load_transnet(weights_dir=None, tf_gpu=None):
    """Khởi tạo TransNetV2 1 lần, tái dùng cho mọi video."""
    configure_tf_gpu(TF_GPU if tf_gpu is None else tf_gpu)

    from transnetv2 import TransNetV2

    if weights_dir:
        try:
            return TransNetV2(model_dir=weights_dir)
        except TypeError:
            # Bản inference cũ không nhận model_dir -> tự tìm weights cạnh module.
            pass
    return TransNetV2()


def detect_shots(video_path, model, threshold=SHOT_THRESHOLD):
    """Trả [(start_frame, end_frame), ...] theo frame index."""
    video_frames = single_pred = all_pred = None
    try:
        video_frames, single_pred, all_pred = model.predict_video(video_path)
        scenes = model.predictions_to_scenes(single_pred, threshold=threshold)
        return [(int(s), int(e)) for s, e in scenes]
    finally:
        # Bắt buộc: không xoá thì RAM leo dần qua từng video.
        del video_frames, single_pred, all_pred

## 5. Stage: keyframe — 7 frame/shot, BaSSL vẫn thấy 3

Công thức gốc trong `services/ingest` lấy 3 frame:

```python
off = int(max(e - s, 0) * inset)
kf  = (s + off, (s + e) // 2, e - off)
```

Lấy `N` điểm **cách đều từ `s+off` đến `e-off`** thì với `N=7`, index `0/3/6` trùng
khít 3 frame trên: `lo+hi = (s+off)+(e-off) = s+e`, nên `floor((lo+hi)/2) = (s+e)//2`
đúng bằng mid cũ. BaSSL nhận input y hệt → gộp scene không đổi. Cell *Kiểm tra* ở
dưới assert lại điều này bằng số thật.

Đã brute-force 120.000 cặp `(s, e)`: **0 lệch** với `N` = 3/5/7/9/11. Nhưng `N` phải
**lẻ** — `N` chẵn thì không có frame nào nằm đúng giữa shot, mid lệch ở ~98% shot và
BaSSL sẽ gộp scene khác bản gốc. Config có `assert` chặn việc này.

Keyframe đặt tên theo **frame index toàn cục** `{frame:06d}.webp` — BE parse số này
rồi cộng/trừ để lấy frame lân cận (`get_neighbor_frames`), nên không đổi sang đánh
số tuần tự. Ghi **1 pass decode tuần tự** và dedup: frame nằm ở nhiều shot chỉ ghi 1 lần.

In [ ]:
import csv

import numpy as np

KEYFRAME_COLS = [f"kf{i}" for i in range(N_KEYFRAMES)]


def keyframe_indices(s, e, n=None, inset=None):
    """n frame cách đều trong [s+off, e-off]. n=7 -> index 0/3/6 = start/mid/end cũ."""
    n = N_KEYFRAMES if n is None else n
    inset = INSET if inset is None else inset

    s, e = int(s), int(e)
    off = int(max(e - s, 0) * inset)
    lo, hi = s + off, e - off
    if hi < lo:                      # shot quá ngắn, inset ăn hết span
        lo = hi = (s + e) // 2
    if n <= 1:
        return [int(np.clip((lo + hi) // 2, s, e))]
    # int() = floor, khớp đúng phép // của công thức gốc.
    return [int(np.clip(int(lo + (hi - lo) * i / (n - 1)), s, e)) for i in range(n)]


def extract_keyframes(video_path, shots, kf_dir, *, fps, width, height,
                      out_h=None, quality=None):
    """Ghi N keyframe/shot vào kf_dir + shots.csv. Trả [{"frame","timestamp"}] đã dedup+sort."""
    import cv2

    out_h = KEYFRAME_HEIGHT if out_h is None else out_h
    quality = WEBP_QUALITY if quality is None else quality

    os.makedirs(kf_dir, exist_ok=True)
    out_w = round(width * out_h / height / 2) * 2   # giữ aspect ratio, số chẵn

    need, rows = {}, []
    for sid, (s, e) in enumerate(shots):
        s, e = int(s), int(e)
        kf = keyframe_indices(s, e)
        for i in kf:
            need[i] = True
        rows.append([sid, s, e, round(s / fps, 3), round(e / fps, 3), *kf])

    with open(os.path.join(kf_dir, "shots.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["shot_id", "start_frame", "end_frame", "start_ts", "end_ts", *KEYFRAME_COLS])
        w.writerows(rows)

    # 1 pass decode tuần tự cả video, chỉ ghi frame nào cần.
    fsz = out_w * out_h * 3
    proc = subprocess.Popen(
        ["ffmpeg", "-i", video_path, "-f", "rawvideo", "-pix_fmt", "bgr24",
         "-s", f"{out_w}x{out_h}", "-v", "quiet", "pipe:1"],
        stdout=subprocess.PIPE, bufsize=fsz * 8,
    )

    n, left, written = 0, len(need), set()
    try:
        while left > 0:
            buf = proc.stdout.read(fsz)
            if len(buf) < fsz:
                break
            if n in need:
                img = np.frombuffer(buf, np.uint8).reshape(out_h, out_w, 3)
                path = os.path.join(kf_dir, f"{n:06d}.webp")
                if cv2.imwrite(path, img, [cv2.IMWRITE_WEBP_QUALITY, quality]):
                    written.add(n)
                else:
                    print(f"    ! ghi hỏng frame {n}")
                left -= 1
            n += 1
    finally:
        proc.stdout.close()
        proc.wait()

    return [{"frame": i, "timestamp": round(i / fps, 3)} for i in sorted(written)]

## 6. Stage: gộp shot → scene (BaSSL CRN)

Mỗi shot → feature ResNet50 trung bình trên **3 keyframe** → Transformer học ngữ cảnh
chuỗi → phân loại nhị phân ranh giới giữa 2 shot liền kề.

`_extract_shot_features` đọc cột `BASSL_KF_COLS` = `("kf0","kf3","kf6")` thay vì
`("kf0","kf1","kf2")` của bản gốc — vẫn đúng 3 ảnh start/mid/end, shape không đổi.

In [ ]:
import pandas as pd


def boundaries_to_scenes(shots_df, boundary_probs, threshold=None):
    """Gom shot thành scene từ xác suất ranh giới của (num_shots - 1) khoảng trống."""
    threshold = SCENE_THRESHOLD if threshold is None else threshold

    scenes = []
    start_frame = int(shots_df.iloc[0]["start_frame"])
    start_ts = float(shots_df.iloc[0]["start_ts"])

    for i, prob in enumerate(boundary_probs):
        if prob > threshold:
            scenes.append({
                "start_frame": start_frame,
                "end_frame": int(shots_df.iloc[i]["end_frame"]),
                "start_time": start_ts,
                "end_time": float(shots_df.iloc[i]["end_ts"]),
                "confidence": round(float(prob), 4),
            })
            start_frame = int(shots_df.iloc[i + 1]["start_frame"])
            start_ts = float(shots_df.iloc[i + 1]["start_ts"])

    scenes.append({
        "start_frame": start_frame,
        "end_frame": int(shots_df.iloc[-1]["end_frame"]),
        "start_time": start_ts,
        "end_time": float(shots_df.iloc[-1]["end_ts"]),
        "confidence": 1.0,
    })
    return scenes


def _build_crn():
    from torch import nn

    class BaSSL_CRN(nn.Module):
        """ResNet50 feature (2048) -> projection 512 -> Transformer -> boundary prob."""

        def __init__(self, input_dim=2048, hidden_dim=512):
            super().__init__()
            self.projection = nn.Linear(input_dim, hidden_dim)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=hidden_dim, nhead=8, batch_first=True)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
            self.boundary_classifier = nn.Linear(hidden_dim, 1)
            self.sigmoid = nn.Sigmoid()

        def forward(self, x):          # x: (1, num_shots, 2048)
            import torch

            feats = self.projection(x)
            feats = self.transformer(feats).squeeze(0)
            boundary_feats = torch.abs(feats[:-1] - feats[1:])
            logits = self.boundary_classifier(boundary_feats).squeeze(-1)
            return self.sigmoid(logits)

    return BaSSL_CRN


def load_resnet_backbone(device):
    """ResNet50 (ImageNet) bỏ lớp fc -> feature 2048."""
    import torch
    from torchvision.models import ResNet50_Weights, resnet50

    backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    backbone.fc = torch.nn.Identity()
    return backbone.eval().to(device)


def load_bassl(checkpoint_path, device):
    """Nạp BaSSL CRN từ .ckpt. Thiếu file -> lỗi rõ ràng, không chạy random weights."""
    import torch

    if not checkpoint_path or not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"BaSSL checkpoint không tồn tại: {checkpoint_path!r}. "
            "Random weights cho scene là vô nghĩa — sửa BASSL_CKPT ở cell Config.")

    model = _build_crn()().to(device)
    state = torch.load(checkpoint_path, map_location=device)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model.load_state_dict(state, strict=False)   # strict=False: bỏ qua key lệch version
    return model.eval()


def _extract_shot_features(shots_df, kf_dir, backbone, device, cols=None):
    """Feature (num_shots, 2048): ResNet50 trung bình trên 3 keyframe/shot."""
    import torch
    import torchvision.transforms as T
    from PIL import Image

    cols = BASSL_KF_COLS if cols is None else cols

    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    feats = []
    with torch.no_grad():
        for _, row in shots_df.iterrows():
            imgs = []
            for col in cols:
                p = os.path.join(kf_dir, f"{int(row[col]):06d}.webp")
                if os.path.exists(p):
                    imgs.append(transform(Image.open(p).convert("RGB")))
                else:
                    imgs.append(torch.zeros(3, 224, 224))
            batch = torch.stack(imgs).to(device)
            feats.append(backbone(batch).mean(dim=0).cpu())
    return torch.stack(feats)


def group_shots_into_scenes(shots_df, kf_dir, *, model, backbone, device, threshold=None):
    """Chạy BaSSL CRN trên feature các shot rồi gom thành scene."""
    import torch

    threshold = SCENE_THRESHOLD if threshold is None else threshold

    if len(shots_df) == 0:
        return []
    if len(shots_df) == 1:
        return boundaries_to_scenes(shots_df, [], threshold)

    features = _extract_shot_features(shots_df, kf_dir, backbone, device)
    with torch.no_grad():
        probs = model(features.unsqueeze(0).to(device))
    return boundaries_to_scenes(shots_df, [p.item() for p in probs], threshold)

## 7. Stage: ASR (chunkformer)

`endless_decode(return_timestamps=True)` transcribe long-form cả video kèm mốc thời gian.

**Cảnh báo**: format timestamp của chunkformer đang **giả định** `"HH:MM:SS:ms"`.
Cell *Kiểm tra* in ra vài segment đầu để bạn đối chiếu bằng mắt trước khi chạy cả batch —
nếu lệch, sửa `_to_seconds`.

In [ ]:
def load_asr_model(name=None):
    from chunkformer import ChunkFormerModel

    return ChunkFormerModel.from_pretrained(name or ASR_MODEL)


def _to_seconds(value):
    """Chuẩn hoá timestamp về giây. Chịu được "HH:MM:SS:ms", "HH:MM:SS.ms", hoặc số giây."""
    if isinstance(value, (int, float)):
        return float(value)
    s = str(value).strip()
    if ":" in s:
        parts = s.split(":")
        if len(parts) == 4:          # HH:MM:SS:ms
            h, m, sec, ms = parts
            return int(h) * 3600 + int(m) * 60 + int(sec) + int(ms) / 1000.0
        if len(parts) == 3:          # HH:MM:SS(.ms)
            h, m, sec = parts
            return int(h) * 3600 + int(m) * 60 + float(sec)
    return float(s)


def _normalize_segments(result):
    """Ép output endless_decode về list[{"start","end","text"}] (giây)."""
    if isinstance(result, str):
        return [{"start": 0.0, "end": 0.0, "text": result.strip()}]

    segments = []
    for seg in result:
        if not isinstance(seg, dict):
            continue
        text = (seg.get("text") or seg.get("decode") or "").strip()
        if not text:
            continue
        segments.append({
            "start": _to_seconds(seg.get("start", 0)),
            "end": _to_seconds(seg.get("end", seg.get("start", 0))),
            "text": text,
        })
    return segments


def transcribe(video_path, model, **decode_kwargs):
    """Transcribe cả video -> list[{"start","end","text"}]."""
    params = {
        "chunk_size": 64,
        "left_context_size": 128,
        "right_context_size": 128,
        "total_batch_duration": 14400,   # giây; đủ cho video dài
        "return_timestamps": True,
    }
    params.update(decode_kwargs)
    return _normalize_segments(model.endless_decode(audio_path=video_path, **params))


def assign_script_to_scenes(scenes, segments):
    """Gán `script` cho mỗi scene = nối text các segment có midpoint nằm trong scene."""
    for sc in scenes:
        lo, hi = float(sc["start_time"]), float(sc["end_time"])
        sc["script"] = " ".join(
            seg["text"] for seg in segments
            if lo <= (seg["start"] + seg["end"]) / 2.0 <= hi
        ).strip()

## 8. Stage: cắt clip scene

Re-encode libx264/crf23 thay vì stream copy, vì stream copy sẽ lệch về keyframe gần
nhất chứ không cắt đúng mốc thời gian.

In [ ]:
def cut_scenes(video_path, scenes, sc_dir):
    """Cắt mỗi scene thành scene_{i:03d}.mp4 trong sc_dir, gán sc["scene_file"]."""
    os.makedirs(sc_dir, exist_ok=True)

    for i, sc in enumerate(scenes):
        start_time = float(sc["start_time"])
        duration = float(sc["end_time"]) - start_time
        fname = f"scene_{i:03d}.mp4"
        if duration <= 0:
            print(f"    ! scene {i} có duration {duration:.3f}s, bỏ qua")
            sc["scene_file"] = None
            continue
        cmd = [
            "ffmpeg", "-y", "-v", "quiet",
            "-ss", str(start_time), "-i", video_path, "-t", str(duration),
            "-c:v", "libx264", "-preset", "fast", "-crf", "23",
            "-c:a", "aac", os.path.join(sc_dir, fname),
        ]
        try:
            subprocess.run(cmd, check=True)
            sc["scene_file"] = fname
        except subprocess.CalledProcessError as ex:
            print(f"    ! lỗi cắt scene {i}: {ex}")
            sc["scene_file"] = None

## 9. Đường dẫn output + URL S3

`Videos_L21_a/video/L21_V001.mp4` → group `L21_a`, video `L21_V001`.

URL trong JSON dựng từ **key đã biết trước**, không phụ thuộc upload đã chạy chưa —
nên `DO_UPLOAD=False` vẫn ghi ra JSON hoàn chỉnh.

In [ ]:
import json


def group_of(video_path):
    """Videos_L21_a/video/L21_V001.mp4 -> ("L21_a", "L21_V001")."""
    name = os.path.splitext(os.path.basename(video_path))[0]
    vdir = os.path.dirname(video_path)
    # Cấu trúc mong đợi: <group_dir>/video/<file>.mp4
    gdir = (os.path.basename(os.path.dirname(vdir))
            if os.path.basename(vdir).lower() == "video"
            else os.path.basename(vdir))
    group = gdir[len("Videos_"):] if gdir.startswith("Videos_") else gdir
    return group, name


def out_paths(group, name):
    """(kf_dir, sc_dir, tr_path) cho 1 video."""
    return (
        os.path.join(OUT_ROOT, f"Keyframes_{group}", "keyframes", name),
        os.path.join(OUT_ROOT, f"Keyscence_{group}", "keyscence", name),
        os.path.join(OUT_ROOT, f"Transcripts_{group}", "transcripts", f"{name}.json"),
    )


def s3_key(local_path):
    """Key trên S3 = đường dẫn tương đối so với OUT_ROOT (mirror y nguyên cây)."""
    return os.path.relpath(local_path, OUT_ROOT).replace("\\", "/")


def s3_url(local_path):
    return f"{S3_URL_BASE}/{s3_key(local_path)}"


def dump_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# Thử nhanh trên video tìm được ở preflight (bỏ qua nếu chưa chạy cell đó).
if globals().get("videos_found"):
    _g, _n = group_of(videos_found[0])
    _kf, _sc, _tr = out_paths(_g, _n)
    print(f"group={_g}  video={_n}")
    print(f"  keyframes  -> {_kf}")
    print(f"  scenes     -> {_sc}")
    print(f"  transcript -> {_tr}")
    print(f"  url mẫu    -> {s3_url(os.path.join(_kf, '000012.webp'))}")

## 10. S3

Lỗi 1 file không chặn các file còn lại. Upload chỉ chạy **sau khi mọi file của video
đã ghi xong**, để không đẩy rác nửa vời lên S3 nếu có stage nào lỗi.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

_s3_client = None


def get_client():
    """boto3 S3 client, lazy singleton (đọc AWS_* từ env)."""
    global _s3_client
    if _s3_client is None:
        import boto3

        ak, sk = os.environ.get("AWS_ACCESS_KEY"), os.environ.get("AWS_SECRET_KEY")
        region = os.environ.get("AWS_REGION")
        _s3_client = (boto3.client("s3", aws_access_key_id=ak, aws_secret_access_key=sk,
                                   region_name=region)
                      if ak and sk else boto3.client("s3", region_name=region))
    return _s3_client


def upload_file(local_path, key):
    get_client().upload_file(local_path, os.environ["AWS_BUCKET_NAME"], key)
    return key


def upload_many(pairs, max_workers=8):
    """Upload song song [(local_path, key)]. Trả list key, None ở vị trí lỗi."""
    if not pairs:
        return []
    get_client()   # khởi tạo trước khi mở pool, tránh nhiều thread cùng tạo client

    results = [None] * len(pairs)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(upload_file, lp, k): i for i, (lp, k) in enumerate(pairs)}
        for fut in as_completed(futs):
            i = futs[fut]
            try:
                results[i] = fut.result()
            except Exception as err:
                print(f"    ! lỗi upload {pairs[i][0]}: {err}")
    return results


def upload_tree(*dirs_or_files):
    """Upload mọi file trong các thư mục/file đã cho, key = đường dẫn tương đối OUT_ROOT."""
    pairs = []
    for target in dirs_or_files:
        if not target or not os.path.exists(target):
            continue
        if os.path.isfile(target):
            pairs.append((target, s3_key(target)))
            continue
        for root, _dirs, files in os.walk(target):
            for fname in files:
                lp = os.path.join(root, fname)
                pairs.append((lp, s3_key(lp)))

    keys = upload_many(pairs)
    ok = sum(k is not None for k in keys)
    print(f"    S3: {ok}/{len(pairs)} file")
    return dict(zip((p[0] for p in pairs), keys))

## 11. Xử lý 1 video

Thứ tự: probe → shot → keyframe → scene → ASR → cắt clip → ghi 3 JSON → upload.

`scenes.json` ghi **cuối cùng** trong nhóm metadata và chính nó là cờ resume, nên một
video chết giữa đường sẽ được chạy lại từ đầu ở lần sau thay vì bị coi là đã xong.

In [ ]:
def process_video(video_path, *, asr_model, transnet_model, bassl_model, backbone, device,
                  upload=None):
    """Chạy 1 video end-to-end. Trả dict thống kê."""
    upload = DO_UPLOAD if upload is None else upload
    group, name = group_of(video_path)
    kf_dir, sc_dir, tr_path = out_paths(group, name)

    width, height, fps, _ = probe(video_path)

    # 1) Shot boundaries -> 2) N keyframe/shot + shots.csv
    shots = detect_shots(video_path, transnet_model, threshold=SHOT_THRESHOLD)
    keyframes = extract_keyframes(video_path, shots, kf_dir,
                                  fps=fps, width=width, height=height)

    # 3) Gộp shot -> scene (BaSSL đọc 3 trong N cột kf*)
    shots_df = pd.read_csv(os.path.join(kf_dir, "shots.csv"))
    scenes = group_shots_into_scenes(shots_df, kf_dir, model=bassl_model,
                                      backbone=backbone, device=device,
                                      threshold=SCENE_THRESHOLD)

    # 4) ASR cả video -> gán script cho từng scene theo thời gian
    segments = transcribe(video_path, asr_model)
    assign_script_to_scenes(scenes, segments)

    # 5) Cắt clip mp4 mỗi scene
    cut_scenes(video_path, scenes, sc_dir)

    # 6) Ghi 3 JSON. *_url là URL S3 đầy đủ; tên file local suy từ frame/scene_id.
    for kf in keyframes:
        kf["keyframe_url"] = s3_url(os.path.join(kf_dir, f"{kf['frame']:06d}.webp"))

    scenes_out = [
        {
            "scene_id": i,
            "script": sc.get("script", ""),
            "start_frame": sc["start_frame"],
            "end_frame": sc["end_frame"],
            "start_time": sc["start_time"],
            "end_time": sc["end_time"],
            "scene_url": (s3_url(os.path.join(sc_dir, sc["scene_file"]))
                          if sc.get("scene_file") else None),
        }
        for i, sc in enumerate(scenes)
    ]

    dump_json(tr_path, {"video_id": name, "language": "vi", "segments": segments})
    dump_json(os.path.join(kf_dir, "keyframes.json"), keyframes)
    dump_json(os.path.join(sc_dir, "scenes.json"), scenes_out)   # cờ resume, ghi cuối

    # 7) Upload sau khi mọi file đã ghi xong
    if upload:
        upload_tree(kf_dir, sc_dir, tr_path)
        if DELETE_LOCAL_AFTER_UPLOAD:
            shutil.rmtree(kf_dir, ignore_errors=True)
            shutil.rmtree(sc_dir, ignore_errors=True)
            print("    đã xoá local (giữ transcript)")

    return {
        "group": group, "video": name,
        "shots": len(shots), "keyframes": len(keyframes),
        "scenes": len(scenes_out), "segments": len(segments),
        "kf_dir": kf_dir, "sc_dir": sc_dir, "tr_path": tr_path,
    }

## 12. Nạp model + chạy batch

Model nặng nạp **đúng 1 lần** rồi tái dùng cho mọi video. Một video lỗi chỉ in ra
rồi đi tiếp, không chặn cả batch.

### Về 2 GPU

Pipeline này **không** song song hoá theo GPU: mọi tensor PyTorch đi về `cuda:{TORCH_GPU}`,
không có `DataParallel`/`DistributedDataParallel`, không chia batch theo device. Cho
2 GPU thì GPU thứ hai **nằm không** — không lỗi, nhưng cũng không nhanh hơn.

Hai thứ đáng làm khi có 2 GPU:

1. **Tách TF khỏi PyTorch**: đặt `TF_GPU = 1` ở cell Config. TransNetV2 (TensorFlow)
   chiếm riêng GPU 1, PyTorch giữ GPU 0 → hết hẳn rủi ro tranh VRAM.
2. **Shard theo session**: mở 2 notebook, một cái `SHARD, NUM_SHARDS = 0, 2`, cái kia
   `1, 2`. Mỗi session có accelerator riêng, không phải sửa logic gì. Đây mới là cách
   thực sự tăng throughput.

Lưu ý nút thắt thật có thể **không phải GPU**: `extract_keyframes` decode tuần tự cả
video bằng ffmpeg trên CPU, và Kaggle chỉ có 4 vCPU. Đo `seconds` ở bảng tổng kết
trước khi tối ưu GPU.

In [ ]:
import time

import torch

device = torch.device(f"cuda:{TORCH_GPU}" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"PyTorch device = {device}   (thấy {n_gpu} GPU)")
if n_gpu > 1 and TF_GPU is None:
    print(f"  ! Có {n_gpu} GPU nhưng TF_GPU=None -> TF dùng chung GPU với PyTorch.\n"
          f"    Đặt TF_GPU=1 ở cell Config để tách ra, GPU 1 đang không dùng gì.")

# TransNetV2 nạp TRƯỚC: nó kéo theo TensorFlow, để lỗi GPU/VRAM lộ ra ngay thay vì
# sau khi đã chờ tải xong model ASR.
t0 = time.time()
transnet_model = load_transnet(TRANSNET_WEIGHTS)
print(f"  TransNet nạp xong ({time.time() - t0:.0f}s)")

t0 = time.time()
asr_model = load_asr_model(ASR_MODEL)
print(f"  ASR      nạp xong ({time.time() - t0:.0f}s)")

t0 = time.time()
backbone = load_resnet_backbone(device)
bassl_model = load_bassl(BASSL_CKPT, device)
print(f"  BaSSL    nạp xong ({time.time() - t0:.0f}s)")

if torch.cuda.is_available():
    for i in range(n_gpu):
        free, total = torch.cuda.mem_get_info(i)
        print(f"  GPU{i}: còn trống {free / 2**30:.1f}/{total / 2**30:.1f} GB")

In [ ]:
videos = videos_found
if NUM_SHARDS > 1:
    videos = [v for i, v in enumerate(videos) if i % NUM_SHARDS == SHARD]
    print(f"shard {SHARD}/{NUM_SHARDS}: {len(videos)}/{len(videos_found)} video")
videos = videos[: LIMIT] if LIMIT else videos
print(f"{len(videos)} video cần xử lý\n")

results, failures = [], []
for i, video_path in enumerate(videos, 1):
    group, name = group_of(video_path)
    _kf, sc_dir, _tr = out_paths(group, name)
    if os.path.exists(os.path.join(sc_dir, "scenes.json")) and not OVERWRITE:
        print(f"[{i}/{len(videos)}] {name} — đã có, bỏ qua")
        continue

    t0 = time.time()
    try:
        res = process_video(video_path,
                            asr_model=asr_model, transnet_model=transnet_model,
                            bassl_model=bassl_model, backbone=backbone, device=device)
        res["seconds"] = round(time.time() - t0, 1)
        results.append(res)
        print(f"[{i}/{len(videos)}] {name}: {res['shots']} shot, "
              f"{res['keyframes']} keyframe, {res['scenes']} scene, "
              f"{res['segments']} segment ({res['seconds']}s)")
    except Exception as ex:
        failures.append((name, repr(ex)))
        print(f"[{i}/{len(videos)}] {name} — LỖI: {ex}")

print(f"\nxong {len(results)} video, lỗi {len(failures)}")
for name, err in failures:
    print(f"  {name}: {err}")

## 13. Kiểm tra

Ba thứ đáng kiểm bằng số, không kiểm bằng mắt:

1. **7 keyframe không làm lệch BaSSL** — cột `kf0/kf3/kf6` phải trùng đúng công thức
   3 frame gốc.
2. Số file `.webp` khớp `keyframes.json`.
3. Timestamp ASR khớp thời gian scene (cái này phải xem bằng mắt, vì `_to_seconds`
   đang giả định format).

In [ ]:
# 1. Assert 7 frame giữ nguyên 3 frame BaSSL, so với công thức gốc của repo.
def _kf3_goc(s, e, inset=INSET):
    off = int(max(e - s, 0) * inset)
    return [int(np.clip(i, s, e)) for i in (s + off, (s + e) // 2, e - off)]


lech = 0
for s, e in [(0, 5), (0, 7), (10, 11), (0, 100), (37, 41), (500, 1234), (7, 7), (0, 1)]:
    new = keyframe_indices(s, e)
    got = [new[int(c[2:])] for c in BASSL_KF_COLS]
    want = _kf3_goc(s, e)
    flag = "OK  " if got == want else "LỆCH"
    if got != want:
        lech += 1
    print(f"  {flag} shot({s:>4},{e:>4})  7frame={new}  ->{got}  gốc={want}")

assert lech == 0, f"{lech} trường hợp lệch — BaSSL sẽ nhận input khác bản gốc"
print("\n1. OK — 3 frame BaSSL trùng khít công thức gốc")

In [ ]:
# 2. Đếm file thật vs metadata.
if not results:
    print("chưa có video nào chạy xong ở lần này — bỏ qua")
else:
    r = results[0]
    kf_json = json.load(open(os.path.join(r["kf_dir"], "keyframes.json"), encoding="utf-8"))
    webps = [f for f in os.listdir(r["kf_dir"]) if f.endswith(".webp")]
    shots_df = pd.read_csv(os.path.join(r["kf_dir"], "shots.csv"))
    mp4s = [f for f in os.listdir(r["sc_dir"]) if f.endswith(".mp4")]
    sc_json = json.load(open(os.path.join(r["sc_dir"], "scenes.json"), encoding="utf-8"))

    print(f"video {r['video']}")
    print(f"  webp trên đĩa      : {len(webps)}")
    print(f"  keyframes.json     : {len(kf_json)}")
    print(f"  shots.csv          : {len(shots_df)} shot x {N_KEYFRAMES} = "
          f"{len(shots_df) * N_KEYFRAMES} (dedup -> {len(kf_json)})")
    print(f"  scene mp4 / json   : {len(mp4s)} / {len(sc_json)}")

    assert len(webps) == len(kf_json), "số webp khác keyframes.json"
    assert len(kf_json) <= len(shots_df) * N_KEYFRAMES, "keyframe nhiều hơn mức tối đa"
    assert set(shots_df.columns) >= set(KEYFRAME_COLS), f"shots.csv thiếu cột {KEYFRAME_COLS}"
    print("\n2. OK")
    print("\nkeyframes.json[0]:")
    print(json.dumps(kf_json[0], ensure_ascii=False, indent=2))
    print("\nscenes.json[0]:")
    print(json.dumps(sc_json[0], ensure_ascii=False, indent=2))

In [ ]:
# 3. Timestamp ASR — phải xem bằng mắt. _to_seconds đang GIẢ ĐỊNH "HH:MM:SS:ms".
if not results:
    print("chưa có video nào chạy xong — bỏ qua")
else:
    r = results[0]
    tr = json.load(open(r["tr_path"], encoding="utf-8"))
    sc_json = json.load(open(os.path.join(r["sc_dir"], "scenes.json"), encoding="utf-8"))

    print(f"{len(tr['segments'])} segment. 5 cái đầu:")
    for seg in tr["segments"][:5]:
        print(f"  [{seg['start']:7.2f} -> {seg['end']:7.2f}]  {seg['text'][:70]}")

    print("\n3 scene đầu — script có khớp thời gian không?")
    for sc in sc_json[:3]:
        print(f"  scene {sc['scene_id']} [{sc['start_time']:7.2f} -> {sc['end_time']:7.2f}]  "
              f"{(sc['script'] or '(rỗng)')[:70]}")

    dur = max((s["end"] for s in tr["segments"]), default=0)
    vid_end = max((s["end_time"] for s in sc_json), default=0)
    print(f"\nsegment cuối kết thúc {dur:.1f}s, video dài ~{vid_end:.1f}s")
    if dur > vid_end * 1.5 or (vid_end > 0 and dur < vid_end * 0.3):
        print("!! lệch nhiều — _to_seconds có thể đang parse sai format, kiểm tra lại")
    else:
        print("phạm vi thời gian hợp lý")

## 14. Tổng kết + xác nhận S3

In [ ]:
if results:
    df = pd.DataFrame(results)[
        ["group", "video", "shots", "keyframes", "scenes", "segments", "seconds"]]
    print(df.to_string(index=False))
    print(f"\ntổng: {df['keyframes'].sum()} keyframe, {df['scenes'].sum()} scene, "
          f"{df['seconds'].sum():.0f}s")

if os.path.exists(OUT_ROOT):
    total = sum(os.path.getsize(os.path.join(dp, f))
                for dp, _dn, fn in os.walk(OUT_ROOT) for f in fn)
    print(f"dung lượng {OUT_ROOT}: {total / 2**20:.1f} MB")
    print(f"đĩa còn trống: {shutil.disk_usage('/kaggle/working').free / 2**30:.1f} GB")

print("\ncây output:")
for root, dirs, files in os.walk(OUT_ROOT):
    depth = root[len(OUT_ROOT):].count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    webp = sum(f.endswith(".webp") for f in files)
    mp4 = sum(f.endswith(".mp4") for f in files)
    other = [f for f in files if not f.endswith((".webp", ".mp4"))]
    extra = []
    if webp:
        extra.append(f"{webp} webp")
    if mp4:
        extra.append(f"{mp4} mp4")
    extra += other
    print("  " * depth + os.path.basename(root) + "/" + ("   " + ", ".join(extra) if extra else ""))

In [ ]:
# Xác nhận key trên S3 đúng cây (chỉ chạy khi đã upload).
if not DO_UPLOAD:
    print("DO_UPLOAD=False — chưa upload gì. Bật lên rồi chạy lại cell batch.")
elif not results:
    print("chưa có video nào chạy xong.")
else:
    prefix = f"Keyframes_{results[0]['group']}/"
    resp = get_client().list_objects_v2(
        Bucket=os.environ["AWS_BUCKET_NAME"], Prefix=prefix, MaxKeys=10)
    objs = resp.get("Contents", [])
    print(f"{resp.get('KeyCount', 0)} object đầu dưới {prefix} "
          f"(tổng còn nữa: {resp.get('IsTruncated')})")
    for o in objs:
        print(f"  {o['Key']}  {o['Size'] / 1024:.0f} KB")
    if objs:
        print(f"\nURL kiểm tra: {S3_URL_BASE}/{objs[0]['Key']}")